# Few-shot Dataset Import to S3 Vector store

This notebook demonstrates how to import the FATURA2 dataset into S3 Vectors for use with the examples-provider Lambda function.

The FATURA2 dataset contains invoice documents that can be used as few-shot examples for document extraction tasks.

## Process Overview:

1. **Load FATURA2 Dataset** - Download and process the dataset
2. **Generate Embeddings** - Create multimodal embeddings using Amazon Nova
3. **Upload to S3 Vectors** - Store embeddings and metadata in S3 Vectors index
4. **Verify Import** - Test similarity search functionality

> **Note**: This notebook requires AWS credentials with permissions for Bedrock, S3, and S3 Vectors services.

## 1. Install Dependencies

In [ ]:
# Let's make sure that modules are autoreloaded
%load_ext autoreload
%autoreload 2

ROOTDIR="../.."
# First uninstall existing package (to ensure we get the latest version)
%pip uninstall -y idp_common

# Install the IDP common package with all components in development mode
%pip install -q -e "{ROOTDIR}/lib/idp_common_pkg[dev, all]"

# Note: We can also install specific components like:
# %pip install -q -e "{ROOTDIR}/lib/idp_common_pkg[ocr,classification,extraction,evaluation]"

# Check installed version
%pip show idp_common | grep -E "Version|Location"

# Install required packages
%pip install -q pillow requests tqdm pandas

# Optionally use a .env file for environment variables
try:
    from dotenv import load_dotenv
    load_dotenv()  
except ImportError:
    pass

## 2. Import Libraries

In [3]:
import json
import zipfile
import requests
from pathlib import Path
from typing import Dict, List, Any
from tqdm import tqdm
import pandas as pd

import boto3
from PIL import Image

# Import IDP common modules
from idp_common import bedrock

print("Libraries imported successfully")

## 3. Configure S3 Vectors and Bedrock

In [ ]:
# Configuration - Update these values based on your deployment of the 'notebooks/examples/dynamic-few-shot-lambda' stack
S3_BUCKET_FOR_IMAGES = "genaiidp-dynamic-few-shot-dynamicfewshotdatasetbuc-nuz4jeue5hds" # Stack output 'DynamicFewShotDatasetBucket'
S3_VECTORS_BUCKET = "genaiidp-dynamic-few-shot"
S3_VECTORS_INDEX = "documents"

EMBEDDING_MODEL_ID = "amazon.nova-2-multimodal-embeddings-v1:0"
EMBEDDING_DIMENSIONS = 3072

# Initialize clients
s3vectors_client = boto3.client('s3vectors')
s3_client = boto3.client('s3')
bedrock_client = bedrock.BedrockClient()

print(f"Configured for S3 Vectors bucket: {S3_VECTORS_BUCKET}")
print(f"Configured for S3 Vectors index: {S3_VECTORS_INDEX}")
print(f"Using embedding model: {EMBEDDING_MODEL_ID}")

## 4. Load FATURA2 Dataset

In [ ]:
# Download and extract FATURA2 dataset from Zenodo
print("Downloading FATURA2 dataset...")

# Configuration for this dataset
IMAGE_VARIANT = 'colored_images'
ANNOTATION_VARIANT = 'Original_Format'
CLASS_LABEL = 'invoice'

# Create datasets directory
datasets_dir = Path('datasets')
datasets_dir.mkdir(exist_ok=True)

# Download the zip file
zip_url = 'https://zenodo.org/records/10371464/files/FATURA2.zip?download=1'
zip_path = datasets_dir / 'FATURA2.zip'

if not zip_path.exists():
    response = requests.get(zip_url, stream=True)
    response.raise_for_status()
    
    with open(zip_path, 'wb') as f:
        for chunk in tqdm(response.iter_content(chunk_size=8192), desc='Downloading'):
            f.write(chunk)
    print(f"Downloaded {zip_path}")
else:
    print(f"Using existing {zip_path}")

# Extract the zip file
extract_dir = datasets_dir / 'invoices_dataset_final'
if not extract_dir.exists():
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(datasets_dir)
    print(f"Extracted to {extract_dir}")
else:
    print(f"Using existing {extract_dir}")

colored_images = extract_dir / IMAGE_VARIANT

# Load images from extracted directory
image_files = list(colored_images.glob('**/*.jpg'))
print(f"Found {len(image_files)} {IMAGE_VARIANT} files")

# Show sample
if image_files:
    sample_image = Image.open(image_files[0])
    print(f"Sample image: {image_files[0].name}")
    print(f"Image size: {sample_image.size}")

print(f"Image variant: {IMAGE_VARIANT}")
print(f"Annotation variant: {ANNOTATION_VARIANT}")
print(f"Class label: {CLASS_LABEL}")

## 5. Process Dataset and Generate Embeddings

In [20]:
def upload_image_to_s3(image_bytes: bytes, s3_key: str) -> str:
    """Upload image to S3 and return S3 URI."""
    s3_client.put_object(
        Bucket=S3_BUCKET_FOR_IMAGES,
        Key=s3_key,
        Body=image_bytes,
        ContentType='image/jpeg'
    )
    return f"s3://{S3_BUCKET_FOR_IMAGES}/{s3_key}"

def load_split(extract_dir, split_name):
    csv_path = extract_dir / (split_name + ".csv")
    return pd.read_csv(csv_path)

def read_annotation(extract_dir, annot_path):
    json_path = extract_dir / "Annotations" / ANNOTATION_VARIANT / annot_path
    with open(json_path, "r") as f:
        annotation = f.read()
    return json.loads(annotation)

def load_image(extract_dir, img_path):
    image_path = extract_dir / IMAGE_VARIANT / img_path
    with open(image_path, "rb") as f:
        image_content = f.read()
    return image_content

def map_labels(annotations):
    labels = {}
    labels['invoice_number'] = annotations.get("NUMBER", {}).get("text", "null").split("\n")
    labels['invoice_date'] = annotations.get("DATE", {}).get("text", "null").split("\n")
    labels['due_date'] = annotations.get("DUE_DATE", {}).get("text", "null").split("\n")
    labels['vendor_name'] = annotations.get("SELLER_NAME", {}).get("text", "null").split("\n")
    labels['vendor_address'] = annotations.get("SELLER_ADDRESS", {}).get("text", "null").split("\n")
    BUYER = annotations.get("BUYER", {}).get("text", "null").split("\n")
    labels['customer_name'] = BUYER[0] if len(BUYER) > 0 else []
    labels['customer_address'] = BUYER[1:] if len(BUYER) > 1 else []
    labels['items'] = "null"
    labels['quantities'] = "null"
    labels['unit_prices'] = "null"
    labels['subtotal'] = annotations.get("SUB_TOTAL", {}).get("text", "null").split("\n")
    labels['tax'] = annotations.get("TAX", {}).get("text", "null").split("\n")
    labels['total_amount'] = annotations.get("TOTAL", {}).get("text", "null").split("\n")
    labels['payment_terms'] = annotations.get("NOTE", {}).get("text", "null").split("\n")
    labels['po_number'] = annotations.get("GSTIN_BUYER", {}).get("text", "null").split("\n")
    return labels

def get_attributes_prompt(labels):
    attributes_prompt = f"""expected attributes are:
        "invoice_number": {", ".join(labels['invoice_number'])}
        "invoice_date": {", ".join(labels['invoice_date'])}
        "due_date": {", ".join(labels['due_date'])}
        "vendor_name": {", ".join(labels['vendor_name'])}
        "vendor_address": {", ".join(labels['vendor_address'])}
        "customer_name": {labels['customer_name']}
        "customer_address": {", ".join(labels['customer_address'])}
        "items": {labels['items']}
        "quantities": {labels['quantities']}
        "unit_prices": {labels['unit_prices']}
        "subtotal": {", ".join(labels['subtotal'])}
        "tax": {", ".join(labels['tax'])}
        "total_amount": {", ".join(labels['total_amount'])}
        "payment_terms": {", ".join(labels['payment_terms'])}
        "po_number": {", ".join(labels['po_number'])}
    """.strip()
    return attributes_prompt

def create_metadata(annotations: Dict, s3_image_uri: str) -> Dict:
    """Create metadata for S3 Vectors entry."""
    class_prompt = f"This is an example of the class '{CLASS_LABEL}'"

    labels = map_labels(annotations)
    attributes_prompt = get_attributes_prompt(labels)

    return {
        "classLabel": CLASS_LABEL,
        "classPrompt": class_prompt,
        "attributesPrompt": attributes_prompt,
        "imagePath": s3_image_uri,
    }

print("Helper functions defined")

## 6. Import Dataset to S3 Vectors

In [ ]:
# Process a subset of the dataset (adjust as needed)
MAX_SAMPLES = 100  # Adjust this number based on your needs
BATCH_SIZE = 10    # Adjust this number based on your needs

dataset_split = load_split(extract_dir, "strat1_train")
samples_to_process = min(MAX_SAMPLES, len(dataset_split))

print(f"Processing {samples_to_process} samples from FATURA2 dataset...")

vectors_to_upload = []
failed_samples = []

for i in tqdm(range(samples_to_process), desc="Processing samples"):
    try:
        df_image = dataset_split.iloc[i]

        # Load annotations
        annotations = read_annotation(extract_dir, df_image["annot_path"])
        
        # Load image
        image_bytes = load_image(extract_dir, df_image["img_path"])

        # Upload image to S3
        s3_key = f"fatura2/{IMAGE_VARIANT}/{df_image['img_path']}"
        s3_image_uri = upload_image_to_s3(image_bytes, s3_key)
        
        # Generate embedding
        embedding = bedrock_client.generate_embedding(
            image_source=image_bytes,
            model_id=EMBEDDING_MODEL_ID,
            dimensions=EMBEDDING_DIMENSIONS
        )
        
        # Create metadata
        metadata = create_metadata(annotations, s3_image_uri)

        # Prepare vector for upload
        vector_entry = {
            "key": f"fatura2_sample_{i:06d}",
            "data": {"float32": embedding},
            "metadata": metadata
        }

        vectors_to_upload.append(vector_entry)
        
        # Upload in batches to avoid memory issues
        if len(vectors_to_upload) >= BATCH_SIZE:  # Batch size
            print(f"\nUploading batch of {len(vectors_to_upload)} vectors...")
            response = s3vectors_client.put_vectors(
                vectorBucketName=S3_VECTORS_BUCKET,
                indexName=S3_VECTORS_INDEX,
                vectors=vectors_to_upload
            )
            print(f"Batch upload response: {response.get('ResponseMetadata', {}).get('HTTPStatusCode')}")
            vectors_to_upload = []  # Clear batch
            
    except Exception as e:
        print(f"\nFailed to process sample {i}: {e}")
        failed_samples.append(i)
        continue

# Upload remaining vectors
if vectors_to_upload:
    print(f"\nUploading final batch of {len(vectors_to_upload)} vectors...")
    response = s3vectors_client.put_vectors(
        vectorBucketName=S3_VECTORS_BUCKET,
        indexName=S3_VECTORS_INDEX,
        vectors=vectors_to_upload
    )
    print(f"Final batch upload response: {response.get('ResponseMetadata', {}).get('HTTPStatusCode')}")

print(f"\nImport completed!")
print(f"Successfully processed: {samples_to_process - len(failed_samples)} samples")
print(f"Failed samples: {len(failed_samples)}")
if failed_samples:
    print(f"Failed sample indices: {failed_samples[:10]}...")  # Show first 10

## 7. Verify Import with Similarity Search

In [ ]:
# Test similarity search with a sample from the dataset
test_split = load_split(extract_dir, "strat1_test")

test_sample_index = 0
df_image = test_split.iloc[test_sample_index]

test_image_bytes = load_image(extract_dir, df_image["img_path"])

print(f"Testing similarity search with sample {extract_dir / IMAGE_VARIANT / df_image['img_path']}...")

# Generate embedding for test image
test_embedding = bedrock_client.generate_embedding(
    image_source=test_image_bytes,
    model_id=EMBEDDING_MODEL_ID,
    dimensions=EMBEDDING_DIMENSIONS
)

# Query S3 Vectors for similar examples
response = s3vectors_client.query_vectors(
    vectorBucketName=S3_VECTORS_BUCKET,
    indexName=S3_VECTORS_INDEX,
    queryVector={"float32": test_embedding},
    topK=5,
    returnDistance=True,
    returnMetadata=True
)

print(f"\nFound {len(response['vectors'])} similar examples:")
for i, vector in enumerate(response['vectors']):
    distance = vector.get('distance', 'N/A')
    key = vector.get('key', 'N/A')
    metadata = vector.get('metadata', {})
    class_label = metadata.get('classLabel', 'N/A')
    class_prompt = metadata.get('classPrompt', 'N/A')
    attributes_prompt = metadata.get('attributesPrompt', 'N/A')
    image_path = metadata.get('imagePath', 'N/A')
    
    print(f"  {i+1}. Key: {key}")
    print(f"     Distance: {distance:.4f}")
    print(f"     Class Label: {image_path}")
    print(f"     Class Prompt: {class_prompt}")
    print(f"     Attributes Prompt: {attributes_prompt}")
    print(f"     Image Path: {image_path}")
    print()

## 8. Summary and Next Steps

In [34]:
print("=== Few-shot Dataset Import Summary ===")
print(f"✅ Dataset: FATURA2 (Invoice documents)")
print(f"✅ Samples processed: {samples_to_process - len(failed_samples)}")
print(f"✅ S3 Vectors Bucket: {S3_VECTORS_BUCKET}")
print(f"✅ S3 Vectors Index: {S3_VECTORS_INDEX}")
print(f"✅ Images stored in: s3://{S3_BUCKET_FOR_IMAGES}/fatura2/{IMAGE_VARIANT}/")
print(f"✅ Embedding Model: {EMBEDDING_MODEL_ID}")
print(f"✅ Similarity search verified")

print("\n=== Next Steps ===")
print("1. Upload your own datasets into S3 Vectors")
print("2. Configure your IDP extraction to use the examples provider Lambda ARN")
print("3. Test document processing with few-shot examples!")